# Principal Component Analysis (PCA)

PCA helps reduce the number of features while keeping the most important information, making it easier to analyze and visualize data.

In [5]:
# STEP 0: Generate Data
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

data = {
    "Math": [80, 85, 78, 92, 70],
    "Physics": [78, 82, 76, 88, 72],
    "Chemistry": [65, 70, 60, 75, 58],
    "Biology": [60, 65, 58, 68, 55],
    "History": [70, 75, 72, 78, 65]
}
df = pd.DataFrame(data)
display(Markdown("## STEP 0: Generate Data"))
display(df)

# STEP 1: Standardize the Data
standardized_rows = []

for idx, row in df.iterrows():
    standardized_row = {}
    for col in df.columns:
        col_values = df[col]
        mean = col_values.sum() / len(col_values)
        variance = ((col_values - mean) ** 2).sum() / (len(col_values) - 1)
        std = variance ** 0.5
        z = (row[col] - mean) / std
        standardized_row[col] = z
    standardized_rows.append(standardized_row)

df_standardized = pd.DataFrame(standardized_rows)
display(Markdown("## STEP 1: Standardized Data"))
display(df_standardized)

# STEP 2: Compute the Covariance Matrix
n_samples = len(df_standardized)

cov_matrix = []

for col1 in df_standardized.columns:
    row = []
    for col2 in df_standardized.columns:
        x = df_standardized[col1]
        y = df_standardized[col2]
        product_sum = sum((x[i] * y[i]) for i in range(n_samples))
        covariance = product_sum / (n_samples - 1)
        row.append(covariance)
    cov_matrix.append(row)

cov_df = pd.DataFrame(cov_matrix, columns=df_standardized.columns, index=df_standardized.columns)
display(Markdown("## STEP 2: Compute the Covariance Matrix"))
display(cov_df)

# STEP 3: Eigen Decomposition
eigenvalues, eigenvectors = np.linalg.eig(cov_df.values)

eigenvalues = eigenvalues.real
eigenvectors = eigenvectors.real

eigen_df = pd.DataFrame({
    "Eigenvalue": eigenvalues,
    "Explained Variance (%)": 100 * eigenvalues / sum(eigenvalues)})

display(Markdown("## STEP 3: Eigen Decomposition"))
display(eigen_df)

# STEP 4: Sort and Select Principal Components
sorted_indices = np.argsort(eigenvalues)[::-1]
sorted_eigenvalues = eigenvalues[sorted_indices]
sorted_eigenvectors = eigenvectors[:, sorted_indices]

explained_variance_ratio = sorted_eigenvalues / np.sum(sorted_eigenvalues)

df_explained = pd.DataFrame({
    "Eigenvalue": sorted_eigenvalues,
    "Explained Variance (%)": explained_variance_ratio * 100})

display(Markdown("## STEP 4: Sort and Select Principal Components"))
display(df_explained)

X_std = df_standardized.values
principal_components = np.dot(X_std, sorted_eigenvectors)
df_pca = pd.DataFrame(principal_components, columns=[f"PC{i+1}" for i in range(principal_components.shape[1])])
display(df_pca)


# STEP 5: Comparison of Manual vs. Scikit-Learn
from sklearn.decomposition import PCA


pca = PCA()
X_sklearn = pca.fit_transform(df_standardized)
df_sklearn_pca = pd.DataFrame(X_sklearn, columns=[f"PC{i+1}" for i in range(X_sklearn.shape[1])])

aligned_manual = df_pca.copy()

for i in range(df_pca.shape[1]):
    if np.dot(df_pca.iloc[:, i], df_sklearn_pca.iloc[:, i]) < 0:
        aligned_manual.iloc[:, i] *= -1

manual_rounded = aligned_manual.round(5)
sklearn_rounded = df_sklearn_pca.round(5)
difference = (manual_rounded - sklearn_rounded).abs().add_suffix(" (abs diff)")

comparison = pd.concat([
    manual_rounded.add_suffix(" (manual aligned)"),
    sklearn_rounded.add_suffix(" (sklearn)"),
    difference
], axis=1)

display(Markdown("## STEP 5: Comparison of Manual vs. Scikit-Learn"))
display(comparison)


## STEP 0: Generate Data

,Math,Physics,Chemistry,Biology,History
0,80,78,65,60,70
1,85,82,70,65,75
2,78,76,60,58,72
3,92,88,75,68,78
4,70,72,58,55,65


## STEP 1: Standardized Data

,Math,Physics,Chemistry,Biology,History
0,-0.122169,-0.196748,-0.085453,-0.228003,-0.404061
1,0.488678,0.459078,0.626656,0.722011,0.606092
2,-0.366508,-0.524661,-0.797562,-0.608009,0.000000
3,1.343864,1.442817,1.338765,1.292019,1.212183
4,-1.343864,-1.180487,-1.082406,-1.178018,-1.414214


## STEP 2: Compute the Covariance Matrix

...

## STEP 4: Sort and Select Principal Components

,Eigenvalue,Explained Variance (%)
0,4.858497e+00,9.716994e+01
1,1.199064e-01,2.398127e+00
2,1.737679e-02,3.475357e-01
3,4.219662e-03,8.439325e-02
4,-1.183059e-16,-2.366117e-15


,PC1,PC2,PC3,PC4,PC5
0,0.461337,-0.209666,0.100357,0.096568,5.646191e-16
1,-1.297141,-0.023220,-0.217338,0.023499,7.355643e-16
2,1.031525,0.591873,0.031864,-0.002055,-1.473261e-16
3,-2.966256,-0.078556,0.104048,-0.054910,1.693173e-15
4,2.770535,-0.280431,-0.018932,-0.063102,1.743878e-15


## STEP 5: Comparison of Manual vs. Scikit-Learn

,PC1 (manual aligned),PC2 (manual aligned),PC3 (manual aligned),PC4 (manual aligned),PC5 (manual aligned),PC1 (sklearn),PC2 (sklearn),PC3 (sklearn),PC4 (sklearn),PC5 (sklearn),PC1 (abs diff),PC2 (abs diff),PC3 (abs diff),PC4 (abs diff),PC5 (abs diff)
0,-0.46134,-0.20967,-0.10036,-0.09657,0.0,-0.46134,-0.20967,-0.10036,-0.09657,0.0,0.0,0.0,0.0,0.0,0.0
1,1.29714,-0.02322,0.21734,-0.02350,0.0,1.29714,-0.02322,0.21734,-0.02350,0.0,0.0,0.0,0.0,0.0,0.0
2,-1.03153,0.59187,-0.03186,0.00206,-0.0,-1.03153,0.59187,-0.03186,0.00206,0.0,0.0,0.0,0.0,0.0,0.0
3,2.96626,-0.07856,-0.10405,0.05491,0.0,2.96626,-0.07856,-0.10405,0.05491,0.0,0.0,0.0,0.0,0.0,0.0
4,-2.77054,-0.28043,0.01893,0.06310,0.0,-2.77054,-0.28043,0.01893,0.06310,0.0,0.0,0.0,0.0,0.0,0.0
